In [1]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
import time
from datasets import load_dataset
import faiss

In [7]:
index = faiss.read_index("/content/drive/MyDrive/books.index")

print("Number of vectors:", index.ntotal)

Number of vectors: 16559


In [2]:
dataset = load_dataset("textminr/cmu-book-summaries")
print(dataset)
books = dataset["train"]

summaries = books["summary"]

print("Number of books:", len(summaries))
print("Example:")
print(summaries[0])

README.md:   0%|          | 0.00/505 [00:00<?, ?B/s]

data/train-00000-of-00001-793a356a05565f(…): reconstructing file:   0%|          |  0.00B / 26.3MB            

data/train-00000-of-00001-793a356a05565f(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16559 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'author', 'pub_year', 'summary'],
        num_rows: 16559
    })
})
Number of books: 16559
Example:
 Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he compares the humans to parasites and teaches the animals a revolutionary song, 'Beasts of England'. When Major dies, two young pigs, Snowball and Napoleon, assume command and turn his dream into a philosophy. The animals revolt and drive the drunken and irresponsible Mr Jones from the farm, renaming it "Animal Farm". They adopt Seven Commandments of Animal-ism, the most important of which is, "All animals are equal". Snowball attempts to teach the animals reading and writing; food is plentiful, and the farm runs smoothly. The pigs elevate themselves to positions of leadership and set aside special food items, ostensibly for their personal health. Napoleon takes the pups from the farm dogs and trains them privately. Napoleon and S

In [3]:
Embedding_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", trust_remote_code=True, device="cuda")
Embedding_model.max_seq_length = 512
reranker = CrossEncoder("BAAI/bge-reranker-base")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [5]:
def create_vector_databse(Embedding_model, summaries):
  embeddings = Embedding_model.encode(
      summaries,
      batch_size=8,
      show_progress_bar=True,
      normalize_embeddings=True
  )
  embeddings = np.asarray(embeddings, dtype="float32")
  embedding_dim = embeddings.shape[1]

  index = faiss.IndexFlatIP(embedding_dim)

  index.add(embeddings)

  print("Number of vectors:", index.ntotal)
  return index

In [6]:
index = create_vector_databse(Embedding_model, summaries)

Batches:   0%|          | 0/2070 [00:00<?, ?it/s]

Number of vectors: 16559


RuntimeError: Error in faiss::FileIOWriter::FileIOWriter(const char*) at /project/faiss/impl/io.cpp:103: Error: 'f' failed: could not open /content/drive/MyDrive/books.index for writing: No such file or directory

In [7]:
faiss.write_index(index, "books.index")
books.save_to_disk("books_metadata")

Saving the dataset (0/1 shards):   0%|          | 0/16559 [00:00<?, ? examples/s]

In [8]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-8B", dtype=torch.float16, device_map="auto")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [11]:
def query2propmt(query):
  query_embedding = Embedding_model.encode(
    [query],
    normalize_embeddings=True
  )

  query_embedding = np.asarray(query_embedding, dtype="float32")
  scores, indices = index.search(query_embedding, 3)
  top_books = []

  for idx, score in zip(indices[0], scores[0]):
      book = books[int(idx)]

      top_books.append({
          "title": book["title"],
          "author": book["author"],
          "pub_year": book["pub_year"],
          "summary": book["summary"],
          "score": float(score)
      })
  context = ""
  for i, book in enumerate(top_books, 1):
      context += f"""
  Book {i}:
  Title: {book['title']}
  Author: {book['author']}
  Publication year: {book['pub_year']}
  Summary: {book['summary']}
  """
  prompt = f"""
  You are a book recommendation assistant.

  The user is looking for:
  {query}

  Here are the three most relevant books retrieved from the book database:

  {context}

  Based only on these books, recommend the most suitable books to the user.

  For each recommendation:
  - Give the title and author.
  - Briefly explain why it matches the user's request.

  If none of the books are a good match, say so instead of inventing recommendations.
  """
  return prompt


In [12]:
def generate_response(prompt):
  messages = [
      {"role": "user", "content": prompt}
  ]
  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False
  )
  inputs = tokenizer( text,return_tensors="pt").to(model.device)
  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=512,
          temperature=0.7,
          do_sample=True
      )

  response = tokenizer.decode(
      outputs[0][inputs["input_ids"].shape[1]:],
      skip_special_tokens=True
  )
  return response

In [13]:
start = time.time()
prompt = query2propmt("hellooooo")
response = generate_response(prompt)
print(response)
print(f"Process took {time.time() - start}")

Hello! It seems your message was just a greeting, and you haven't specified the type of books you're interested in. However, based on the three books retrieved, here are the recommendations:

1. **Golem in the Gears** by Piers Anthony  
   This book is a science fiction novel that blends elements of fantasy and steampunk. It features a golem, a creature from folklore, set in a world of advanced machinery. If you're interested in imaginative, genre-blending stories, this could be a good match.

2. **Clara Vaughan** by R. D. Blackmore  
   Though the plot outline is not fully described, this book is a historical novel set in the 19th century. If you enjoy richly detailed narratives with a focus on character development and historical settings, this might be worth exploring.

3. **Stadium Beyond the Stars** by Stephen Marlowe  
   The title suggests a space-themed adventure, potentially with a focus on sports or futuristic settings. If you're drawn to speculative fiction or stories set in